<a href="https://colab.research.google.com/github/sutida254526/parttime-scheduler/blob/main/app_ver_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install streamlit
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 29.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,

In [5]:
# app.py
import streamlit as st
from app_M2T1P8 import solver_parttime  # ใส่ solver_parttime ไว้ในไฟล์แยก
import random, numpy as np

st.set_page_config(page_title="Part-time Scheduler", page_icon="🗓️", layout="wide")

st.markdown("<h1 style='color: #1E3A8A;'>Part-time Employee Scheduler</h1>", unsafe_allow_html=True)
st.markdown("<p style='color: #374151;'>จัดตารางพนักงานพาร์ทไทม์สำหรับร้านอาหาร</p>", unsafe_allow_html=True)

# 1. กรอกชื่อพนักงาน
employee_names_input = st.text_input("กรอกชื่อพนักงาน (คั่นด้วย ,)")
employee_names = [name.strip() for name in employee_names_input.split(",") if name.strip()]
num_employees = len(employee_names)

# 2. เลือกเดือน
month = st.selectbox("เลือกเดือน", ["มกราคม","กุมภาพันธ์","มีนาคม","เมษายน","พฤษภาคม","มิถุนายน",
                                     "กรกฎาคม","สิงหาคม","กันยายน","ตุลาคม","พฤศจิกายน","ธันวาคม"])

# 3. แบบสุ่มหรือเรียง
schedule_type = st.radio("เลือกวิธีจัดตาราง", ["สุ่ม","เรียง"])

# 4. กรอกเงื่อนไขพนักงาน
st.markdown("### เงื่อนไขพนักงาน (วันและกะที่สะดวก)")
all_employees_avail = {}
for idx, name in enumerate(employee_names, start=1):
    with st.expander(f"{name}"):
        available_days = st.multiselect(f"วันที่ {name} สามารถทำงานได้", ["อาทิตย์","จันทร์","อังคาร","พุธ","พฤหัส","ศุกร์"], key=f"days_{idx}")
        available_shifts = st.multiselect(f"กะที่ {name} สามารถทำงานได้", ["1","2","3","4"], key=f"shifts_{idx}")
        # แปลงเป็น dictionary ตัวเลข
        day_map = {"อาทิตย์":1,"จันทร์":2,"อังคาร":3,"พุธ":4,"พฤหัส":5,"ศุกร์":6}
        shift_map = {"1":1,"2":2,"3":3,"4":4}
        emp_avail = {day_map[d]: [shift_map[s] for s in available_shifts] for d in available_days}
        all_employees_avail[idx] = emp_avail

# 5. กรอกจำนวนพนักงานหลักต่อกะ
st.markdown("### จำนวนพนักงานหลักต่อกะ")
W_per_t = {}
for t in range(1,5):
    W_per_t[t] = st.number_input(f"กะ {t} ต้องการพนักงานหลักกี่คน", min_value=0, value=1, step=1, key=f"main_{t}")

# 6. กรอกจำนวนพนักงานสำรองต่อกะ
st.markdown("### จำนวนพนักงานสำรองต่อกะ")
W_backup = {}
for t in range(1,5):
    W_backup[t] = st.number_input(f"กะ {t} ต้องการพนักงานสำรองกี่คน", min_value=0, value=1, step=1, key=f"backup_{t}")

# 7. จำกัดจำนวนกะต่อสัปดาห์
st.markdown("### จำกัดจำนวนกะต่อสัปดาห์")
max_shift = st.number_input("จำนวนกะสูงสุด/สัปดาห์ ต่อคน", min_value=1, value=3)

# 8. พนักงานขาด
st.markdown("### พนักงานขาด (เลือกถ้ามี)")
employee_absent = {}
for idx, name in enumerate(employee_names, start=1):
    with st.expander(f"{name}"):
        absent_days = st.multiselect(f"วันขาดของ {name}", ["อาทิตย์","จันทร์","อังคาร","พุธ","พฤหัส","ศุกร์"], key=f"absent_days_{idx}")
        absent_shifts = st.multiselect(f"กะขาดของ {name}", ["1","2","3","4"], key=f"absent_shifts_{idx}")
        day_map = {"อาทิตย์":1,"จันทร์":2,"อังคาร":3,"พุธ":4,"พฤหัส":5,"ศุกร์":6}
        shift_map = {"1":1,"2":2,"3":3,"4":4}
        employee_absent[idx] = [(day_map[d], shift_map[s]) for d in absent_days for s in absent_shifts]

# ปุ่ม Solve
if st.button("สร้างตารางงาน"):
    data = {
        "num_employees": num_employees,
        "W_per_t": W_per_t,
        "max_shift_i": max_shift,
        "P_idt": all_employees_avail,
        "cost_per_shift": {1:320,2:160,3:160,4:160},
        "absent": employee_absent
    }
    result = solver_parttime(data)

    st.markdown(f"### ค่าใช้จ่ายรวม: {result['total_cost']} บาท/สัปดาห์")

    st.markdown("### ตารางพนักงานหลัก (Main)")
    for d in range(1,7):
        st.write(f"Day {d}")
        for t in range(1,5):
            st.write(f"Shift {t}: {result['main'][(d,t)]}")

    st.markdown("### ตารางพนักงานสำรอง (Backup)")
    for d in range(1,7):
        st.write(f"Day {d}")
        for t in range(1,5):
            st.write(f"Shift {t}: {result['backup'][(d,t)]}")


ModuleNotFoundError: No module named 'app_M2T1P8'